# Session 3 · Why It Feels Like Someone Is There
*Understanding Language Models*

**In this session you will:** talk to a recreation of the first chatbot from the 1960s, look inside it, then talk to a small modern model and experiment with giving it different personalities. The aim is to notice the moment you start feeling there is *someone* on the other side, and to ask what that feeling is made of.

**Time:** about 60–75 minutes. For faster replies, use *Runtime → Change runtime type → T4 GPU*.

### How to use this notebook
- This is a **Google Colab notebook**: a page that mixes reading with small pieces of code you can run.
- To run a grey code box, click it and press **Shift + Enter** (or click the ▶ button on its left).
- **Run the boxes in order, top to bottom.** If something breaks, go to *Runtime → Restart session* and start again from the top.
- You never have to *write* code. Where you see text inside quotation marks, like `"this"`, you can change the words and run the box again. That is the whole skill.
- Boxes marked **Setup** load the machinery. You can open them if you are curious, but you do not need to read them.

## The big idea

In 1966 the computer scientist Joseph Weizenbaum built ELIZA, a program that imitated a therapist by turning your sentences back into questions. It had no understanding at all, just a list of patterns. Weizenbaum was disturbed to find that people, including his own secretary, confided in it and asked to be left alone with it. The tendency to feel a mind behind a machine's words became known as the **Eliza effect**.

A striking argument is that the Eliza effect is no longer an occasional illusion. With modern chatbots it has become a general **condition of our culture**. There is also a subtler point. The early thinkers of AI, like Alan Turing, did not really think machines would *copy* human intelligence. They saw humans and machines as both taking part in shared formal systems (language, logic, games). "Intelligence" then starts to look less like an original that machines imitate and more like what the philosopher Jean Baudrillard called a **simulacrum**: a copy without an original.

Let's start in 1966.

## Part 1 · Talk to ELIZA

In [ ]:
#@title Setup: build a mini ELIZA (instant)
import re, random

REFLECT = {"i": "you", "me": "you", "my": "your", "am": "are", "i'm": "you're",
           "mine": "yours", "myself": "yourself", "you": "I", "your": "my",
           "yours": "mine", "was": "were", "i've": "you've", "i'll": "you'll"}

def reflect(fragment):
    return " ".join(REFLECT.get(w, w) for w in fragment.lower().split())

RULES = [
    (r"i need (.*)",        ["Why do you need {0}?", "Would it really help you to get {0}?"]),
    (r"i feel (.*)",        ["Tell me more about feeling {0}.", "How long have you felt {0}?",
                             "Do you often feel {0}?"]),
    (r"i am (.*)|i'm (.*)", ["How long have you been {0}?", "Why do you say you are {0}?",
                             "How does being {0} make you feel?"]),
    (r"i can'?t (.*)",      ["What makes you think you can't {0}?", "Have you tried?"]),
    (r"because (.*)",       ["Is that the real reason?", "What else comes to mind?"]),
    (r"(.*)\bmy (mother|father|family|sister|brother|son|daughter)(.*)",
                            ["Tell me more about your {1}.", "How do you get on with your {1}?"]),
    (r"(.*) art(.*)|(.*) music(.*)|(.*) paint(.*)",
                            ["What does your work mean to you?", "When did you first start making work?"]),
    (r"(.*)\?",             ["Why do you ask that?", "What do you think?",
                             "Perhaps you already know the answer."]),
    (r"(hello|hi|hey)(.*)", ["Hello. What is on your mind today?"]),
    (r"(.*)",               ["Please tell me more.", "I see. Go on.", "How does that make you feel?",
                             "Let's explore that further.", "Why do you think that is?"]),
]

def eliza(text):
    text = text.lower().strip().rstrip(".!")
    for pattern, answers in RULES:
        m = re.match(pattern, text)
        if m:
            groups = [reflect(g) for g in m.groups() if g is not None] or [""]
            reply = random.choice(answers)
            try:
                return reply.format(*groups, *([""] * 3))
            except IndexError:
                return reply
    return "Please go on."

def chat_with_eliza():
    print("ELIZA: Hello. I am ELIZA. What would you like to talk about? (type 'bye' to stop)")
    while True:
        you = input("YOU:   ")
        if you.lower().strip() in ("bye", "quit", "exit"):
            print("ELIZA: Goodbye. It was nice talking to you."); break
        print("ELIZA:", eliza(you))

print("ELIZA is ready.")

In [ ]:
chat_with_eliza()

Talk to it for three or four minutes. Tell it about a project, a worry, your week. Type `bye` to finish.

**Notice:** at what point, if any, did it feel like it was listening? What did *you* contribute to making the conversation work?

## Part 2 · Look inside

Here is everything ELIZA "knows". Run the box.

In [ ]:
for pattern, answers in RULES:
    print(f"IF you write something like  {pattern!r}")
    print(f"   THEN reply with one of    {answers}\n")

About ten rules and a trick for swapping "my" into "your". The sense of being heard was produced almost entirely by **genre**: the therapeutic conversation, where short open questions and reflecting back are exactly what we expect. Weizenbaum picked the one genre where an empty listener sounds wise.

**Try:** add your own rule. Copy the format of a line in `RULES` in the setup box (for example, a rule for sentences containing "deadline"), rerun the setup, and talk again.

## Part 3 · Talk to a modern (small) model

Now a genuine language model. This one is tiny compared with ChatGPT, Gemini or Claude, and it makes mistakes, which is useful here because its habits are easier to see.

In [ ]:
#@title Setup: load a small chat model (takes 1–3 minutes the first time)
# A small, openly available chat model. It is far weaker than ChatGPT or Claude,
# which is useful: its habits and defaults are easier to see.
import torch, textwrap
from transformers import AutoTokenizer, AutoModelForCausalLM

CHAT_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32
chat_tok = AutoTokenizer.from_pretrained(CHAT_MODEL)
chat_model = AutoModelForCausalLM.from_pretrained(CHAT_MODEL).to(device=device, dtype=dtype)

def ask(prompt, system="You are a helpful assistant.", temperature=0.8,
        max_new_tokens=250, show=True):
    """Send one message to the chat model and return its reply."""
    msgs = [{"role": "system", "content": system},
            {"role": "user", "content": prompt}]
    text = chat_tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    enc = chat_tok(text, return_tensors="pt").to(device)
    out = chat_model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=True,
                              temperature=temperature, top_p=0.9,
                              pad_token_id=chat_tok.eos_token_id)
    reply = chat_tok.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    if show:
        for para in reply.split("\n"):
            print(textwrap.fill(para, 90) if para.strip() else "")
    return reply

print(f"Chat model ready (running on {device}).")

In [ ]:
ask("I run a small community arts space and I'm worried we won't survive another year. What should I do?")

In [ ]:
# Ask it anything. Change the text in quotes.
ask("What is the difference between art and craft?")

**Compare with ELIZA:** both are producing text by pattern. What is different about the *experience*? Did you catch yourself thanking it, or feeling reassured?

## Part 4 · Personality is a setting

Chat models are given a hidden instruction called a **system prompt** that sets their persona. Change it, and the "someone" you are talking to changes too. Same question, three characters:

In [ ]:
question = "What do you think of my idea to turn an old cinema into a community theatre?"

personas = {
    "Enthusiastic friend": "You are an enthusiastic, supportive friend who loves the arts.",
    "Cautious funder":     "You are a cautious arts funder who asks hard questions about money and risk.",
    "Poet":                "You are a poet. Answer everything in a short free-verse poem.",
}
for name, system in personas.items():
    print(f"\n==== {name} ====")
    ask(question, system=system, max_new_tokens=150)

Write your own persona below: a museum guide, a grandmother who remembers the old cinema, a hostile critic.

In [ ]:
ask("Tell me about the paintings in this room.",
    system="You are a friendly museum guide at a small gallery of contemporary West African painting.")

**Notice:** the guide will happily describe paintings it cannot see and a gallery that does not exist. It is performing the *genre* of the museum guide, just as ELIZA performed the genre of the therapist. The persona feels like a character; underneath, it is one instruction line.

## Discussion

1. Where did the feeling of "someone there" come from: the machine, the genre, or you?
2. Many cultural organisations are considering chatbots for visitor services, education or archives. Given what you saw in Part 4, what should visitors be told? What would you never want a bot to do on your behalf?
3. One suggestion is that intelligence may be a "copy without an original". Is human conversation also partly genre and performance? Does that make it less real?

## Glossary
- **ELIZA**: the first chatbot (1966), built from simple pattern rules.
- **Eliza effect**: attributing understanding or feeling to a machine because of its words.
- **System prompt**: hidden instructions that set a chatbot's role and tone.
- **Simulacrum**: Baudrillard's term for a copy that has no original.

## Going further
- Joseph Weizenbaum, *Computer Power and Human Reason* (1976), his later critique of his own creation.
- Alan Turing, "Computing Machinery and Intelligence" (1950), the "imitation game" paper.